In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [2]:
x, y = make_classification(n_samples = 10000, n_features = 10, n_informative = 3, random_state = 1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

In [3]:
# Train Decision Tree first and see the result

dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)
y_pred = dt.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.9535


# Bagging

In [4]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 100,
    max_samples = 0.25,
    bootstrap = True,    # Sampling with replacement
    random_state = 1
)

bag.fit(x_train, y_train)
y_pred = bag.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.9665


In [5]:
# See, we took 0.25 means 25% samples in training. So it should be 2000 rows. Lets ensure this

bag.estimators_samples_    # This shows samples of each of 100 Decision Trees

[array([4228, 6364, 4362, ..., 2190, 4333,  150]),
 array([2783, 6100, 2169, ..., 6319, 3743, 7892]),
 array([7902, 7853, 1615, ..., 4250, 6094, 5968]),
 array([1922, 4177, 1447, ..., 2232, 2736, 4881]),
 array([1414, 1378, 6402, ...,   38, 4943, 6790]),
 array([6421, 7051, 6798, ...,  530, 3416, 7477]),
 array([1101, 5659, 5434, ..., 7465, 2012, 2659]),
 array([2580, 7740, 5157, ...,  629, 5670, 1967]),
 array([2179, 6495, 5012, ..., 2340, 1709, 7858]),
 array([  77, 2636, 3753, ..., 6191, 7446, 7375]),
 array([7315, 1219, 4653, ..., 1227, 1176, 5000]),
 array([1343, 2406, 5425, ..., 7596, 7859, 5841]),
 array([5665, 1447,  619, ..., 4654, 6244,   40]),
 array([1291, 4193, 6117, ..., 5037, 4373,   51]),
 array([1218, 1787, 4397, ..., 7480, 3999,  489]),
 array([5070, 6542,  582, ..., 2790, 2334, 6019]),
 array([2877,  686, 4033, ..., 1829, 3782, 1146]),
 array([ 931, 1904,  628, ..., 6027, 7536, 1269]),
 array([1730, 3961, 7832, ..., 7938, 5856, 3810]),
 array([ 266,  120, 2825, ..., 

In [6]:
print(bag.estimators_samples_[0])    # This shows samples of first decision tree 
print()
print()

print(bag.estimators_samples_[0].shape)

[4228 6364 4362 ... 2190 4333  150]


(2000,)


# Bagging using SVM

In [7]:
bag = BaggingClassifier(
    estimator = SVC(),
    n_estimators = 100,
    max_samples = 0.25,
    bootstrap = True,    # Sampling with replacement
    random_state = 1
)

bag.fit(x_train, y_train)
y_pred = bag.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.956


# Pasting
Here we take samples without replacement

In [8]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 100,
    max_samples = 0.25,
    bootstrap = False,    # Sampling without replacement
    random_state = 1,
    verbose = 1,
    n_jobs = -1
)

bag.fit(x_train, y_train)
y_pred = bag.predict(x_test)

print(accuracy_score(y_test, y_pred))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    6.9s remaining:   20.9s
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    7.1s finished
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    0.0s remaining:    0.0s


0.9675


[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    0.0s finished


# Random Subspace
Here, we only do column sampling(With or without replacement), we don't do row sampling here.

In [9]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 100,
    max_samples = 0.25,
    bootstrap = False,    # Sampling without replacement
    random_state = 1,
    bootstrap_features = True,    # Column sampling True
    max_features = 0.5    # 50% out of total 10 features
)

bag.fit(x_train, y_train)
y_pred = bag.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.9585


In [10]:
# Let's ensure it took 50% features only.

bag.estimators_features_[0].shape

(5,)

# Random Patches
Here we perform both, row sampling and column sampling

In [11]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 500,
    max_samples = 0.25,
    bootstrap = True,    # Sampling with replacement
    random_state = 1,
    bootstrap_features = True,    # Column sampling True
    max_features = 0.5    # 50% out of total 10 features
)

bag.fit(x_train, y_train)
y_pred = bag.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.9655


In [12]:
# Let's ensure it took 50% features and 25% rows.

print(bag.estimators_samples_[0].shape)    # 2000 rows
print(bag.estimators_features_[0].shape)    # 5 Features

(2000,)
(5,)


# OOB score (Out Of Box score)

- It is statistically proven that whenever we drive sample with replacement, there are 37% samples which doesn't get selected.
- So in Bagging, whenever we go for sampling with replacement, there are high chance that 37% rows are not selected in any dataset.
- Means 37% rows are not seen by any model.
- We can use these rows to check the accuracy of our model.
- This accuracy is called oob score.

- `There is a parameter *oob* which if set to True, returns OOB Score.`

In [13]:
bag = BaggingClassifier(
    estimator = KNeighborsClassifier(),
    n_estimators = 150,
    max_samples = 0.35,
    bootstrap = True,    # With replacement
    bootstrap_features = True,    # Columns with replacement
    max_features = 0.5,    # Take 0.5 features
    oob_score = True,
    n_jobs = -1,
    random_state = 1
)

bag.fit(x_train, y_train)

y_pred = bag.predict(x_test)

print(f"OOB Score: {bag.oob_score_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")    # This will give almost same accuracy as oob score

OOB Score: 0.952
Accuracy: 0.9545


# Bagging Tips

- Bagging generally gives better results than pasting
- Good results come around 25% - 50% max_samples (Row samples)
- Random Subspaces and Random Patches should be only used when dealing with high dimensional data bcz both of these includes Random Features Sampling.
- To find the correct hyperparameter, we can use `RandomSearchCV` or `GridSearchCV` or `Optuna`

# GridSearchCV using Optuna

In [15]:
import optuna
from sklearn.model_selection import cross_val_score

from sklearn.naive_bayes import GaussianNB

In [ ]:
def objective(trial):

    # Suggest values for hyperparameters
    estimator_names = trial.suggest_categorical('estimator', ['DecisionTreeClassifier', 'KNeighborsClassifier', 'GaussianNB'])
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_samples = trial.suggest_float('max_samples', 0.25, 0.5)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    bootstrap_features = trial.suggest_categorical('bootstrap_features', [True, False])
    max_features = trial.suggest_float('max_features', 0.25, 0.5)

    # Mappring estimators: Optuna cannot store objects. So we need to store string in estimator and then map with the objects
    estimator_map = {'DecisionTreeClassifier': DecisionTreeClassifier(),
                     'GaussianNB': GaussianNB(),
                     'KNeighborsClassifier': KNeighborsClassifier()}
    
    estimator = estimator_map[estimator_names]

    bag = BaggingClassifier(
        estimator = estimator,
        n_estimators = n_estimators,
        max_samples = max_samples,
        bootstrap = bootstrap,
        bootstrap_features = bootstrap_features,
        max_features = max_features,
        oob_score = True if bootstrap else False,    # OOB score is impossed without bootstrap. It will give error with bootstrap.
        random_state = 1
    )

    # Performing 5 fold cross validation

    cross_val = cross_val_score(
        estimator = bag,
        X = x_train,
        y = y_train,
        scoring = 'accuracy',
        cv = 3,
        n_jobs = -1
    ).mean()

    return cross_val

In [19]:
study = optuna.create_study(
    direction = 'maximize',
    sampler = optuna.samplers.TPESampler(seed = 1)
    )    # optuna.samplers.TPESampler: by-default

study.optimize(func = objective, n_trials = 50, n_jobs = -1)


# We are saying to optimizer() that this (objective()) is the process of finding accuracy, do it for 50 trials 

# direction = 'maximize' bcz we are using accuracy
# sampler = optuna.samplers.TPESampler() by-default

[I 2026-06-12 14:31:34,344] A new study created in memory with name: no-name-e7c72f25-d999-437e-ac9e-4867851c784d
[I 2026-06-12 14:31:40,718] Trial 0 finished with value: 0.9611251238006352 and parameters: {'estimator': 'DecisionTreeClassifier', 'n_estimators': 378, 'max_samples': 0.3820359761763764, 'bootstrap': False, 'bootstrap_features': False, 'max_features': 0.4489476859590077}. Best is trial 0 with value: 0.9611251238006352.
[I 2026-06-12 14:31:43,645] Trial 5 finished with value: 0.9577497955291223 and parameters: {'estimator': 'DecisionTreeClassifier', 'n_estimators': 213, 'max_samples': 0.31358745274824074, 'bootstrap': True, 'bootstrap_features': False, 'max_features': 0.48564443716189953}. Best is trial 0 with value: 0.9611251238006352.
[I 2026-06-12 14:31:46,581] Trial 3 finished with value: 0.8918759029839949 and parameters: {'estimator': 'KNeighborsClassifier', 'n_estimators': 255, 'max_samples': 0.2958993437277595, 'bootstrap': False, 'bootstrap_features': False, 'max_f

In [20]:
print(f'Best accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_params}')

Best accuracy: 0.9622501707166574
Best hyperparameters: {'estimator': 'DecisionTreeClassifier', 'n_estimators': 427, 'max_samples': 0.4605096090229073, 'bootstrap': False, 'bootstrap_features': False, 'max_features': 0.4362326892376925}


In [34]:
study.best_params

{'estimator': 'DecisionTreeClassifier',
 'n_estimators': 427,
 'max_samples': 0.4605096090229073,
 'bootstrap': False,
 'bootstrap_features': False,
 'max_features': 0.4362326892376925}

In [ ]:
# Let's built BaggingClassifier with these best parameters

bag = BaggingClassifier(**study.best_params)

bag.fit(x_train, y_train)

y_pred = bag.predict(x_test)

cross_val = cross_val_score(
    estimator = bag,
    X = x_train,
    y = y_train,
    scoring = 'accuracy',
    cv = 5,
    n_jobs = -1)

print(f"Cross Validation Score: {cross_val}")

# We are getting error bcz study.best_params stores estimators as string and we need to pass 
# an object of model in estimator of BaggingClassifier()

InvalidParameterError: The 'estimator' parameter of BaggingClassifier must be an object implementing 'fit' and 'predict' or None. Got 'DecisionTreeClassifier' instead.

In [33]:
# Step 1 - Get best params
best = study.best_params

# Step 2 - Map string to actual object
estimator_map = {
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'GaussianNB': GaussianNB(),
    'KNeighborsClassifier': KNeighborsClassifier()
}

best_estimator = estimator_map[best['estimator']]

# Step 3 - Rebuild BaggingClassifier correctly
bag = BaggingClassifier(
    estimator = best_estimator,
    n_estimators = best['n_estimators'],
    max_samples = best['max_samples'],
    bootstrap = best['bootstrap'],
    bootstrap_features = best['bootstrap_features'],
    max_features = best['max_features'],
    oob_score = True if best['bootstrap'] else False,
    random_state = 1
)

bag.fit(x_train, y_train)

y_pred = bag.predict(x_test)

cross_val = cross_val_score(
    estimator = bag,
    X = x_train,
    y = y_train,
    scoring = 'accuracy',
    cv = 5,
    n_jobs = -1).mean()

print(f"Cross Validation Score: {cross_val}")
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")


Cross Validation Score: 0.9615
Accuracy: 0.962
